# Qwen3 on Amazon Bedrock Mantle

Alibaba's Qwen3 family on the `bedrock-mantle` endpoint — a wide range of sizes plus dedicated coder and vision-language variants. This notebook covers the general-purpose models end to end; `02-qwen3-coder-and-vision.ipynb` covers the specialists.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `qwen.qwen3-32b` | Dense 32B — the reliable default; also fine-tunable on mantle |
| `qwen.qwen3-235b-a22b-2507` | MoE 235B total / 22B active — highest quality |
| `qwen.qwen3-next-80b-a3b-instruct` | MoE 80B total / 3B active — cost-efficient |

### Which API? Chat Completions.
This family is served by the **OpenAI-compatible Chat Completions API** on the
`bedrock-mantle` endpoint, at the bare `/v1` path. The Responses API returns
**400 "does not support this API"** for these models — we prove that in §2 rather
than asking you to take it on trust.

### Self-contained, but see also
Everything you need is here. For deeper background on shared mechanics:
- **Auth (SigV4 + short-term API keys), the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR, CloudWatch namespace** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT measurement** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

### Prerequisites
```bash
pip install openai aws-bedrock-token-generator
```
AWS credentials with `bedrock-mantle:CreateInference` and
`bedrock-mantle:CallWithBearerToken` — both granted by the managed policy
`AmazonBedrockMantleInferenceAccess`.

In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from mantle import err, parse_json_lenient, post, ttft

REGION = "us-east-1"

DENSE_32B = "qwen.qwen3-32b"
MOE_235B = "qwen.qwen3-235b-a22b-2507"
MOE_80B = "qwen.qwen3-next-80b-a3b-instruct"


# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("models  :", [DENSE_32B, MOE_235B, MOE_80B])

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
models  : ['qwen.qwen3-32b', 'qwen.qwen3-235b-a22b-2507', 'qwen.qwen3-next-80b-a3b-instruct']


## 1. First call

Auth is a short-term Bedrock API key minted from your ambient IAM credentials.
It expires within 12 hours and **cannot be refreshed** — mint a new one instead.
(`../00-foundations/01` shows the self-refreshing provider and the SigV4
alternative that needs no key at all.)

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token — don't construct one at import time and
# reuse it for hours, because the baked-in key expires.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=DENSE_32B,
    messages=[{"role": "user", "content": "Explain the difference between a dense and a sparse (MoE) transformer, in two sentences."}],
    max_tokens=250,
)
print(completion.choices[0].message.content)
print("\nusage:", completion.usage.model_dump_json())

A dense transformer uses the same set of parameters for every input token, while a sparse (Mixture-of-Experts, MoE) transformer selectively activates only a subset of expert layers for each token, making it more parameter-efficient. This allows MoE transformers to scale to more parameters without increasing computation for every token.

usage: {"completion_tokens":64,"prompt_tokens":32,"total_tokens":96,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

AWS recommends the Responses API for new applications in general — but
availability is per-model. Probe both surfaces so the 400 is visible:

In [3]:
for api_name, path, body in [
    ("Chat Completions", f"{PREFIX}/chat/completions",
     {"model": DENSE_32B, "messages": [{"role": "user", "content": "Reply OK"}],
      "max_tokens": 16}),
    ("Responses (/v1)", f"{PREFIX}/responses",
     {"model": DENSE_32B, "input": "Reply OK", "max_output_tokens": 16}),
    ("Responses (/openai/v1)", "/openai/v1/responses",
     {"model": DENSE_32B, "input": "Reply OK", "max_output_tokens": 16}),
]:
    code, data = post(path, body, region=REGION)
    print(f"  {api_name:24} -> HTTP {code} {'' if code == 200 else err(data)[:64]}")

  Chat Completions         -> HTTP 200 


  Responses (/v1)          -> HTTP 400 The model 'qwen.qwen3-32b' does not support the '/v1/responses' 


  Responses (/openai/v1)   -> HTTP 400 The model 'qwen.qwen3-32b' does not support the '/openai/v1/resp


Concrete consequences of being Chat-Completions-only:

- **You own the conversation history.** There is no `previous_response_id`
  server-side state on this API — send the full `messages` array each turn.
- **Reasoning content is not returned.** `reasoning_effort` is accepted and the
  model does think, but the OpenAI Chat Completions schema has nowhere to put the
  trace, so you pay for those tokens without seeing them.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle — Gemma 4 rejects `top_p`, and Grok rejects `temperature` — so never share
one sampling config across families.

In [4]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {"model": DENSE_32B,
            "messages": [{"role": "user", "content": "Reply OK"}], "max_tokens": 16}
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


Note `max_tokens=1` is accepted here. The Responses API enforces a minimum of
16 — another reason the two surfaces are not interchangeable.

## 4. Streaming

Chat Completions streams `data: {...}` SSE frames carrying
`choices[0].delta.content`, terminated by `data: [DONE]`.

In [5]:
stream = client.chat.completions.create(
    model=DENSE_32B,
    messages=[{"role": "user", "content": "List four practical uses of long-context language models."}],
    max_tokens=300,
    stream=True,
)
chunks = 0
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        chunks += 1
        print(delta, end="", flush=True)
print(f"\n\n[{chunks} content deltas received]")

1. **Document Summarization**  
   Long-context language models can process and summarize lengthy texts such as research papers, legal documents, or business reports, making it easier to extract key points without reading the full content.

2. **Code Generation and Documentation

**  
   These models can understand and generate code from extended documentation or multi-file projects, helping developers write, debug, and document software more efficiently.

3. **Multi

-turn Dialogue Systems**  
   By maintaining long context, these models enhance chat

bots and virtual assistants to handle complex, extended conversations with coherence and context-aware

 responses.

4. **Content Creation and Editing**  
   Long-context models can assist in writing and editing tasks such as novel

 writing, scriptwriting, or article editing by generating consistent and contextually relevant content over thousands

 of words.



[7 content deltas received]


## 5. Multi-turn — you manage the history

No server-side state on this API. Append each turn yourself.

In [6]:
messages = [
    {"role": "system", "content": "You are concise. Two sentences maximum."},
    {"role": "user", "content": "What is quantisation in the context of LLM serving?"},
]
first = client.chat.completions.create(model=DENSE_32B, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "Which quantisation format is most common for GPU inference?"})

second = client.chat.completions.create(model=DENSE_32B, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}")

assistant: Quantisation in LLM serving reduces the precision of model weights (e.g., from 32-bit to 8-bit or lower), decreasing memory usage and computational requirements while maintaining acceptable performance. This enables faster inference and deployment on hardware with limited resources.



assistant: The most common quantisation format for GPU inference is **8-bit integer (INT8)**, as it offers a good balance between performance, accuracy, and hardware compatibility. GPUs are optimized for low-precision arithmetic, making INT8 widely supported and efficient in frameworks like TensorRT and Hugging Face.

input tokens grew: 37 -> 109


That growth is the cost of client-side history. Families on the Responses API can
avoid it with `previous_response_id` (see `../03-google-gemma/`), at the price of
30-day server-side retention.

## 6. Reasoning effort

`reasoning_effort` is accepted. The trace is not returned — but the token count
moves, which is how you can tell the model really is thinking harder.

In [7]:
print(f"{'effort':10} {'status':>7} {'completion tokens':>18}")
print("-" * 38)
for effort in ("none", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": DENSE_32B,
         "messages": [{"role": "user", "content": "A shop sells pens at 3 for $2. How much for 17 pens? Show your working."}],
         "max_tokens": 400, "reasoning_effort": effort},
        region=REGION,
    )
    tokens = (data.get("usage") or {}).get("completion_tokens", "-")
    print(f"  {effort:8} {code:>7} {tokens!s:>18}")

effort      status  completion tokens
--------------------------------------


  none         200                203


  low          200                225


  medium       200                229


  high         200                400


In [8]:
# Qwen3 models often emit an explicit <think>...</think> block in the CONTENT
# when reasoning hard. Strip it before showing the answer to a user.
import re

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": DENSE_32B,
     "messages": [{"role": "user", "content": "What is 17 * 23? Think step by step."}],
     "max_tokens": 500, "reasoning_effort": "high"},
    region=REGION,
)
raw = data["choices"][0]["message"]["content"] or ""
cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.S).strip()
print("has <think> block:", "<think>" in raw)
print("raw first 120 chars:", repr(raw[:120]))
print("cleaned answer   :", repr(cleaned[:160]))

has <think> block: False
raw first 120 chars: ''
cleaned answer   : ''


## 7. Tool use (function calling)

Chat Completions nests the schema under `"function"` — unlike the Responses API,
which puts `name`/`parameters` at the top level. Same concept, different shape.

In [9]:
def lookup_inventory(sku: str, warehouse: str = "main") -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0, "C-300": 7}
    return {"sku": sku, "warehouse": warehouse,
            "quantity": stock.get(sku.upper(), 0),
            "in_stock": stock.get(sku.upper(), 0) > 0}


tools = [{
    "type": "function",
    "function": {
        "name": "lookup_inventory",
        "description": "Look up stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {
                "sku": {"type": "string", "description": "SKU code, e.g. A-100"},
                "warehouse": {"type": "string", "enum": ["main", "overflow"]},
            },
            "required": ["sku"],
        },
    },
}]

convo = [{"role": "user", "content": "Do we have SKU A-100 in stock?"}]
first = client.chat.completions.create(
    model=DENSE_32B, messages=convo, tools=tools, tool_choice="auto", max_tokens=300
)
msg = first.choices[0].message
print("finish_reason:", first.choices[0].finish_reason)
print("tool_calls:", [(c.function.name, c.function.arguments) for c in (msg.tool_calls or [])])

if msg.tool_calls:
    convo.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = lookup_inventory(**args)
        convo.append({"role": "tool", "tool_call_id": call.id,
                       "content": json.dumps(result)})
    final = client.chat.completions.create(
        model=DENSE_32B, messages=convo, tools=tools, max_tokens=200
    )
    print("\nfinal answer:", final.choices[0].message.content)

finish_reason: tool_calls
tool_calls: [('lookup_inventory', '{"sku": "A-100"}')]



final answer: Yes, we have SKU A-100 in stock. There are 42 units available in the main warehouse.


### Forcing a specific tool

`tool_choice` can compel a named function. This is the most portable route to
strict structured output: the arguments *are* your JSON.

**But treat it as best-effort, not a guarantee.** In repeated testing about 1
call in 10 ignored the forced choice and returned prose with
`finish_reason="stop"`. Always check for the tool call and retry.

In [10]:
emit = [{
    "type": "function",
    "function": {
        "name": "emit_review",
        "description": "Return the structured review analysis.",
        "parameters": {
            "type": "object",
            "properties": {
                "summary": {"type": "string"},
                "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
                "would_recommend": {"type": "boolean"},
            },
            "required": ["summary", "sentiment", "would_recommend"],
        },
    },
}]

def emit_review(prompt, model=DENSE_32B, attempts=3):
    """Forced tool call, with a retry.

    IMPORTANT: forcing `tool_choice` is honoured *almost* always, not always.
    In repeated testing roughly 1 call in 10 came back with finish_reason="stop"
    and prose instead of a tool call. Production code must handle that, so this
    helper retries rather than indexing [0] and hoping.
    """
    for attempt in range(attempts):
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            tools=emit,
            tool_choice={"type": "function", "function": {"name": "emit_review"}},
            max_tokens=300,
        )
        choice = completion.choices[0]
        calls = choice.message.tool_calls or []
        if calls:
            if attempt:
                print(f"(succeeded on attempt {attempt + 1})")
            # parse_json_lenient, not json.loads: some models append characters
            # after a well-formed object even in strict modes.
            return parse_json_lenient(calls[0].function.arguments)
        print(f"attempt {attempt + 1}: no tool call "
              f"(finish_reason={choice.finish_reason}) — retrying")
    raise RuntimeError("model would not emit the forced tool call")


review = emit_review("Review: 'Fast delivery, but the packaging arrived crushed.'")
print(json.dumps(review, indent=2))

{
  "summary": "The product had fast delivery, but the packaging was crushed upon arrival.",
  "sentiment": "neutral",
  "would_recommend": false
}


## 8. Structured output with `response_format`

Two variants: loose `json_object`, and schema-enforced `json_schema`.

### Budget enough tokens, or you get nothing

A reasoning-capable model may spend most of its budget thinking before it emits
the opening brace. If `max_tokens` runs out first you get **HTTP 200 with empty
content** and `finish_reason="length"` - not an error, just nothing usable.
Always check `finish_reason` before parsing.

In [11]:
def json_object_call(prompt, max_tokens, model=DENSE_32B):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": model, "messages": [{"role": "user", "content": prompt}],
         "max_tokens": max_tokens, "response_format": {"type": "json_object"}},
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


PROMPT = "Give the capital and population of France as JSON."
for budget in (64, 600):
    code, finish, content = json_object_call(PROMPT, budget)
    print(f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} "
          f"content_len={len(content)}")
    if finish == "length" and not content.strip():
        print("    -> truncated before any JSON was emitted; raise max_tokens")
    elif content.strip():
        print("    ->", parse_json_lenient(content))

max_tokens=  64 HTTP 200 finish=stop     content_len=73
    -> {'country': 'France', 'capital': 'Paris', 'population': 67990000}


max_tokens= 600 HTTP 200 finish=stop     content_len=73
    -> {'country': 'France', 'capital': 'Paris', 'population': 65273511}


In [12]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": DENSE_32B,
     "messages": [{"role": "user", "content": "Describe France."}],
     "max_tokens": 250,
     "response_format": {"type": "json_schema",
                          "json_schema": {"name": "country", "strict": True,
                                           "schema": schema}}},
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")   # may be None!
print("json_schema ->", code, "| finish_reason:", choice.get("finish_reason"))
print("raw:", repr((content or "")[:160]))

if choice.get("finish_reason") == "length":
    # Reasoning consumed the budget before the object closed. Retry bigger.
    print("truncated - retrying with a larger budget")
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": DENSE_32B,
         "messages": [{"role": "user", "content": "Describe France."}],
         "max_tokens": 2000,
         "response_format": {"type": "json_schema",
                              "json_schema": {"name": "country", "strict": True,
                                               "schema": schema}}},
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content")
    print("retry finish_reason:", choice.get("finish_reason"))

parsed = parse_json_lenient(content or "")
print("parsed:", json.dumps(parsed, indent=2))
assert set(parsed) >= {"country", "capital"}, parsed

json_schema -> 200 | finish_reason: stop
raw: '{\n  "country": "France",\n  "capital": "Paris",\n  "population_millions": 65.896412425467985899563299475501668060302734375\n}'
parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 65.89641242546799
}


**Always parse leniently.** Even in strict mode, some mantle models append
characters after a valid object (Gemma 4 does this in ~half of runs), which makes
a bare `json.loads()` raise on output that is otherwise fine.

## 9. Compare the models in this family

Same prompt across three architectures: a dense 32B, a large MoE, and a cost-efficient MoE. Watch latency against quality.

In [13]:
task = "In one sentence, why does a mixture-of-experts model cost less to serve than a dense model of the same total size?"

print(f"{'model':44} {'latency':>9} {'out tok':>8}  answer")
print("-" * 108)
for model in [DENSE_32B, MOE_80B, MOE_235B]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {"model": model, "messages": [{"role": "user", "content": task}],
         "max_tokens": 160},
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:44} {'-':>9} {'-':>8}  HTTP {code}: {err(data)[:40]}")
        continue
    text = (data["choices"][0]["message"]["content"] or "").strip().replace("\n", " ")
    print(f"{model:44} {elapsed:>8.2f}s "
          f"{data['usage']['completion_tokens']:>8}  {text[:44]!r}")

model                                          latency  out tok  answer
------------------------------------------------------------------------------------------------------------


qwen.qwen3-32b                                   1.02s       45  'A mixture-of-experts model costs less to ser'


qwen.qwen3-next-80b-a3b-instruct                 1.80s       71  'A mixture-of-experts (MoE) model costs less '


qwen.qwen3-235b-a22b-2507                        1.36s       48  'A mixture-of-experts (MoE) model costs less '


## 10. Latency: TTFT and throughput

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority — they mostly separate under
contention, so single samples on an idle account look flat.
(`../00-foundations/03` has the full treatment.)

In [14]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {"model": DENSE_32B,
         "messages": [{"role": "user", "content": "List four practical uses of long-context language models."}],
         "max_tokens": 200, "service_tier": tier},
        region=REGION,
    )
    if m.get("error"):
        print(f"{tier:10} {m['error']:>32}  (tier not supported by this model)")
    else:
        print(f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
              f"{m['frames_per_s']:>10.1f}")

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         1.099      1.881        7.7


flex            2.059      2.081       91.7


priority        1.204      1.790       10.2


## 11. Production hardening

Retries, cost attribution, and privacy. Mantle has **no RPM quota** — throttling
is token-based, and most models here have no published TPM quota at all, so
capacity is fair-share. That makes retry-with-backoff mandatory, not optional.

In [15]:
code, project = post(
    "/v1/organization/projects",
    {"name": "qwen3-samples",
     "tags": {"Application": "Qwen3Demo", "Environment": "Demo"}},
    region=REGION,
)
project_id = project.get("id")
print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/chat/completions",
    {"model": DENSE_32B,
     "messages": [{"role": "user", "content": "Reply OK"}], "max_tokens": 16},
    region=REGION,
    headers={"OpenAI-Project": project_id},   # cost attribution
)
print("attributed call ->", code)

project: 200 proj_fv3hkimwvtcpo766qxxx


attributed call -> 200


In [16]:
class Qwen3Client:
    """Production-shaped wrapper for this family on bedrock-mantle."""

    def __init__(self, model=DENSE_32B, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def chat(self, messages, *, max_tokens=512, tools=None, schema=None, effort=None):
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.7,
            "service_tier": self.tier,
        }
        if tools:
            body["tools"] = tools
        if effort:
            body["reasoning_effort"] = effort
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429 + 5xx with exponential backoff and jitter.
        code, data = post(f"{PREFIX}/chat/completions", body,
                          region=self.region, headers=headers)
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    def json(self, prompt, schema, **kw):
        data = self.chat([{"role": "user", "content": prompt}], schema=schema, **kw)
        return parse_json_lenient(data["choices"][0]["message"]["content"])


bot = Qwen3Client(tier="flex", project=project_id)
out = bot.json(
    "Name the largest ocean and its average depth in metres.",
    {"type": "object",
      "properties": {"ocean": {"type": "string"}, "avg_depth_m": {"type": "number"}},
      "required": ["ocean", "avg_depth_m"], "additionalProperties": False},
)
print("structured result:", out)

structured result: {'ocean': 'Pacific Ocean', 'avg_depth_m': 3970}


In [17]:
code, archived = post(f"/v1/organization/projects/{project_id}/archive", {}, region=REGION)
print("archived demo project:", code, archived.get("status"))

archived demo project: 200 archived


## Gotchas — Qwen3 on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` (that's gemma-4 / gpt-5.x / grok) |
| Responses API | Returns **400** for this family — Chat Completions only |
| History | No `previous_response_id`; you send `messages` every turn |
| Reasoning trace | `reasoning_effort` works but the trace is never returned |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Sampling | `temperature` **and** `top_p` both fine here; not true family-wide |
| `max_tokens` | 1 is valid here; Responses API demands ≥16 |
| `content` can be `None` | Check `finish_reason` before slicing/parsing content |
| Quotas | No RPM quota; most models have no published TPM — retry with backoff |
| `reserved` tier | Rejected as a parameter; arranged via your account team |
| CloudWatch | Metrics land in `AWS/BedrockMantle`, not `AWS/Bedrock` |
| `<think>` blocks | Qwen3 can emit reasoning inline in `content` — strip it |
| Fine-tuning | `qwen3-32b` is one of only two mantle-fine-tunable models (us-west-2) |

## Where next
- Same API shape: `../05-deepseek/`, `../06-zai-glm/`, `../09-minimax/`
- Different API shape: `../03-google-gemma/` (Responses),
  `../02-anthropic-claude/` (Messages), `../01-openai-gpt/` (web search, caching)
- Shared mechanics: `../00-foundations/`